# 04 — OCR baseline evaluation and error analysis

This notebook reads local artifacts produced by `scripts/run_ocr.py` and `scripts/evaluate_ocr.py`. It compares full-page transcription annotations with Tesseract word boxes using the documented center-in-polygon alignment. Full-page annotation polygons have coarser granularity than word boxes, so detection F1 is reported as N/A. Receipt images and text remain local; clear outputs before committing.

Run the commands below first from the repository root:

```bash
uv run python scripts/run_ocr.py --split internal_dev --output-dir outputs/ocr/tesseract_internal_dev
uv run python scripts/evaluate_ocr.py --split internal_dev --output-dir outputs/ocr/evaluation
```


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from matplotlib.patches import Polygon
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from docucheck.evaluation.metrics import align_word_predictions_to_lines
from docucheck.ocr.inference import load_manifest

MANIFEST = load_manifest(ROOT / "data/processed/manifest.jsonl")
PREDICTION_ROWS = load_manifest(ROOT / "outputs/ocr/tesseract_internal_dev/predictions.jsonl")
METRICS = json.loads((ROOT / "outputs/ocr/evaluation/metrics.json").read_text(encoding="utf-8"))
PREDICTIONS = {row["image_id"]: row for row in PREDICTION_ROWS}
RECORDS = {row["image_id"]: row for row in MANIFEST if row.get("split") == "internal_dev"}
DATASET_ROOT = Path(os.environ.get("MC_OCR_ROOT", ROOT / "data/mcocr")).expanduser()
print(f"Images with transcription GT: {METRICS['images_with_full_page_ground_truth']}")
print(f"Images with predictions: {METRICS['images_with_ground_truth_and_predictions']}")
print(f"Evaluated GT regions: {METRICS['evaluated_ground_truth_regions']:,}; excluded invalid geometry: {METRICS['excluded_invalid_geometry_regions']:,}")
print(f"CER: {METRICS['cer']:.3f}; WER: {METRICS['wer']:.3f}; detection: {METRICS['detection_metrics']['status']}")
print("Alignment: prediction word-box center inside smallest valid GT polygon; assigned words sorted by y then x.")

## Aggregate quality and image-level error distribution

CER is micro-averaged over normalized characters, and WER uses whitespace-separated Vietnamese syllables. CER/WER can exceed 1 because insertions are counted; many predicted boxes fall outside annotated polygons and some reference regions receive no prediction. These results describe this alignment protocol and are not KIE field extraction scores.

In [ ]:
image_metrics = load_manifest(ROOT / "outputs/ocr/evaluation/per_image_metrics.jsonl")
df = pd.DataFrame(image_metrics)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for column, title, ax in [("cer", "Image-level CER", axes[0]), ("wer", "Image-level WER", axes[1])]:
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    ax.hist(values.clip(upper=5), bins=25, color="#287271", edgecolor="white")
    ax.axvline(values.median(), color="#e76f51", linestyle="--", label=f"median={values.median():.2f}")
    ax.set(title=title + " (values >5 clipped for display)", xlabel="Error rate", ylabel="Images")
    ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["gt_characters"], df["cer"].clip(upper=5), alpha=.55, s=22, color="#e76f51")
ax.set(xlabel="Ground-truth characters per image", ylabel="CER (display clipped at 5)", title="Error versus transcript length")
plt.tight_layout(); plt.show()
print("Image-level groups with fewer than 10 examples are not interpreted separately.")

## Local annotated image gallery

Green polygons show word-level Tesseract predictions; magenta outlines show full-page GT regions. The GT geometry is coarser, and text labels are included only in this local interactive view. A high-error case and a lower-error case are selected from actual evaluation results.

In [ ]:
def show_case(image_id):
    record = RECORDS[image_id]
    path = DATASET_ROOT / record["image_path"]
    image = Image.open(path).convert("RGB")
    pred_row = PREDICTIONS.get(image_id, {})
    predictions = pred_row.get("predictions", []) if pred_row.get("status") == "ok" else []
    gt = [r for r in record.get("ocr_regions", []) if r.get("text_raw") and r.get("polygon") and not r.get("validation_issues")]
    aligned, _ = align_word_predictions_to_lines(predictions, gt)
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.imshow(image)
    for p in predictions:
        ax.add_patch(Polygon(p["polygon"], fill=False, edgecolor="#16a085", linewidth=.8, alpha=.8))
    for region in gt:
        ax.add_patch(Polygon(region["polygon"], fill=False, edgecolor="#d63384", linewidth=1.3, alpha=.85))
    ax.plot([], [], color="#16a085", label="Tesseract word box")
    ax.plot([], [], color="#d63384", label="Full-page GT region")
    ax.legend(loc="upper right")
    ax.set_title(f"Local image {image_id}: {len(predictions)} predictions, {len(gt)} valid GT regions")
    ax.axis("off"); plt.tight_layout(); plt.show()
    comparison = pd.DataFrame([{"GT region": row["text_gt"], "Tesseract aligned text": row["text_pred"], "matched boxes": row["prediction_count"]} for row in aligned])
    display(comparison.head(12))

ordered = sorted((r for r in image_metrics if r.get("cer") is not None), key=lambda r: r["cer"])
chosen = [ordered[len(ordered)//2]["image_id"], ordered[-1]["image_id"]] if ordered else []
for image_id in chosen:
    show_case(image_id)

## Failure Analysis

Observed patterns from the internal development evaluation and local overlays:

1. **Missed annotated regions:** 1,643 of 8,304 valid GT regions received no assigned word boxes. Week 2 should retain an explicit missing-evidence state and assess recall on annotation units compatible with the OCR output.
2. **Spurious boxes outside annotated text:** 12,913 predicted boxes fell outside every valid GT polygon; the gallery also shows isolated predictions away from receipt text. Week 2 should keep confidence/provenance and inspect unmatched boxes before downstream extraction.
3. **Region granularity mismatch:** GT outlines span lines or groups of words while the baseline emits individual word boxes. This makes one-to-one detection scores misleading and complicates recognition alignment. Week 2 should establish compatible line/word annotations or a validated aggregation step before comparing systems.

Detection precision/recall/F1 remains **N/A** for this annotation pairing. OCR results do not measure the future KIE task of extracting seller, address, timestamp, or total fields.
